# Falcon-H1-3B Layer Experiments — Google Colab
Run specific layers of the patch-effect experiment.
Set `START_LAYER` and `END_LAYER` below, then **Runtime → Run all**.

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────────
HF_TOKEN      = ""         # paste your HuggingFace token
START_LAYER   = 0          # first layer to run (inclusive)
END_LAYER     = 7          # last  layer to run (inclusive)
NUM_INSTANCES = 100
NUM_SAMPLES   = 1000
MODEL_ID      = "tiiuae/Falcon-H1-3B-Instruct"
BRANCH        = "fix/colab-oom-8bit"
REPO_URL      = "https://github.com/NitzanZacharia/100rep.git"
OUTPUT_DIR    = "/content/results"
# ───────────────────────────────────────────────────────────────────────────────

In [ ]:
# Verify GPU and decide whether to use 8-bit loading
import torch
assert torch.cuda.is_available(), "No GPU found — change Runtime type to GPU"
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {torch.cuda.get_device_name(0)}  |  VRAM: {vram_gb:.1f} GB")

# T4 has ~15 GB; Falcon-H1-3B float16 weights (~6 GB) + 100-instance Mamba
# activations fill it. Use 8-bit on GPUs with < 20 GB to free ~3 GB.
USE_8BIT = vram_gb < 20
print(f"8-bit loading: {'ON' if USE_8BIT else 'OFF'}")

In [ ]:
# Clone repo
import os
if not os.path.isdir("/content/100rep"):
    !git clone -b {BRANCH} {REPO_URL} /content/100rep
else:
    !git -C /content/100rep fetch && git -C /content/100rep checkout {BRANCH} && git -C /content/100rep pull
%cd /content/100rep

In [ ]:
# Install dependencies
!pip install -q transformers accelerate datasets tqdm matplotlib pandas bitsandbytes
!pip install -q tabulate seaborn scikit-learn tensorboard pyvene

import sys
if "/content/100rep" not in sys.path:
    sys.path.insert(0, "/content/100rep")
if os.path.isdir("/content/100rep/CausalAbstraction"):
    sys.path.append("/content/100rep/CausalAbstraction")

In [ ]:
# Memory allocator (Colab PyTorch is recent enough for expandable_segments)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,garbage_collection_threshold:0.8"

!mkdir -p {OUTPUT_DIR} /content/100rep/logs

# Mount Google Drive early so you can authenticate and walk away
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Run experiment
cmd = (
    f"python run_layer_experiments.py "
    f"--model-id '{MODEL_ID}' "
    f"--output-dir '{OUTPUT_DIR}' "
    f"--num-instances {NUM_INSTANCES} "
    f"--num-samples {NUM_SAMPLES} "
    f"--start-layer {START_LAYER} "
    f"--end-layer {END_LAYER} "
    f"--trust-remote-code "
    + ("--load-in-8bit " if USE_8BIT else "")
    + (f"--hf-token '{HF_TOKEN}' " if HF_TOKEN else "")
)
print("Running:", cmd)
!{cmd}

In [ ]:
# Save results to Google Drive
import shutil, os

DRIVE_DEST = f"/content/drive/MyDrive/falcon-h1-results/layers_{START_LAYER}-{END_LAYER}"

# copytree copies the whole directory tree (including layer_N/ subdirectories)
if os.path.exists(DRIVE_DEST):
    shutil.rmtree(DRIVE_DEST)
shutil.copytree(OUTPUT_DIR, DRIVE_DEST)

print(f"Results saved to Google Drive: {DRIVE_DEST}")
for root, dirs, files in os.walk(DRIVE_DEST):
    for f in files:
        print(" ", os.path.join(root, f).replace(DRIVE_DEST, ""))

In [ ]:
# Download results as a zip
import shutil
zip_path = "/content/results.zip"
shutil.make_archive("/content/results", "zip", OUTPUT_DIR)

from google.colab import files
files.download(zip_path)